In [1]:
import os 


os.chdir("..") # this is to change the current working directory to the parent directory, which is the root of the project.
#os.chdir("student_Performance_p1") 

In [2]:
%pwd

'c:\\Users\\penze\\Desktop\\MLdev_ops\\student_Performance_p1'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestConfig:
  root_dir: Path 
  source_URL: str 
  local_data_file: Path 
  unzip_dir: Path 



In [4]:
from src.student_Performance_p1.constants import * 
from src.student_Performance_p1.utils.common import read_yaml, create_directories


class ConfigurationManager:
    def __init__(self):
        # Read config.yaml
        self.config = read_yaml(CONFIG_FILE_PATH)

        # Read params.yaml
        self.params = read_yaml(PARAMS_FILE_PATH)

        # Read schema.yaml
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        # Create main artifacts folder
        create_directories([self.config.artifacts_root])

    def get_data_ingest_config(self) -> DataIngestConfig:
        # Access data_ingestion section from config.yaml
        config = self.config.data_ingestion

        # Create data ingestion folder
        create_directories([config.root_dir])

        # Create DataIngestConfig object
        data_ingest_config = DataIngestConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingest_config
    
    """
     Here you would normally read the YAML file and parse it into the DataIngestConfig dataclass
     For simplicity, we'll just return a hardcoded config
    we get all this info from the config.yaml file, but for now we are hardcoding it to test the dataclass and the configuration manager class"""

In [5]:
import requests
import tarfile


class DataIngestor:
    def __init__(self, config: DataIngestConfig):
        # Store the data ingestion configuration
        # This config contains:
        # - source_URL
        # - local_data_file
        # - unzip_dir
        self.config = config

    def download_data(self):
        # Download the dataset from the source URL
        response = requests.get(self.config.source_URL)

        # If the URL is wrong or download fails, this will raise an error
        response.raise_for_status()

        # Save the downloaded file locally
        # Example: artifacts/data_ingestion/housing.tgz
        with open(self.config.local_data_file, "wb") as f:
            f.write(response.content)

        print(f"Data downloaded successfully to: {self.config.local_data_file}")

    def unzip_data(self):
        # Your dataset is a .tgz file, not a .zip file
        # So we use tarfile instead of zipfile
        with tarfile.open(self.config.local_data_file, "r:gz") as tar_ref:
            tar_ref.extractall(path=self.config.unzip_dir)

        print(f"Data extracted successfully to: {self.config.unzip_dir}")

In [6]:
try:
    config_manager = ConfigurationManager()

    data_ingest_config = config_manager.get_data_ingest_config()

    print(data_ingest_config)

    data_ingestion = DataIngestor(config=data_ingest_config)

    data_ingestion.download_data()

    data_ingestion.unzip_data()

except Exception as e:
    print(f"Error in data ingestion: {e}")

[2026-05-21 16:58:14,622: INFO: common: YAML file 'config\config.yaml' Loading successfully.]
[2026-05-21 16:58:14,624: INFO: common: YAML file 'params.yaml' Loading successfully.]
[2026-05-21 16:58:14,626: INFO: common: YAML file 'schema.yaml' Loading successfully.]
[2026-05-21 16:58:14,630: INFO: common: Directories created successfully: ['artifacts']]
[2026-05-21 16:58:14,632: INFO: common: Directories created successfully: ['artifacts/data_ingestion']]
DataIngestConfig(root_dir='artifacts/data_ingestion', source_URL='https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.tgz', local_data_file='artifacts/data_ingestion/housing.tgz', unzip_dir='artifacts/data_ingestion')
Data downloaded successfully to: artifacts/data_ingestion/housing.tgz
Data extracted successfully to: artifacts/data_ingestion


C:\Users\penze\AppData\Local\Temp\ipykernel_12244\1098914563.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall(path=self.config.unzip_dir)
